# Demo D: KV Cache Growth + Capacity Planning

**Workshop Part 1, Group 3** | LLM Inference at Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_d_capacity_calculator.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/demos/demo_d_capacity_calculator.ipynb)

**Goal:** Derive KV cache cost from model config, visualize how it scales,
and calculate exactly how many users fit on each GPU.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'matplotlib'])

from transformers import AutoConfig
import matplotlib.pyplot as plt

## 1. Why KV Cache Exists

During decode, attention needs K and V vectors from ALL previous tokens.
- Without cache: recompute everything every step. Gets slower and slower (O(n squared)).
- With cache: store K,V once. Read them back each step. Fast (O(n)) but costs memory.

Every production system uses the KV cache. Let's calculate exactly what it costs.

In [ ]:
# Load real model config to derive KV cache size
config = AutoConfig.from_pretrained('mistralai/Mistral-7B-v0.1')

# Extract the numbers that matter
n_layers = config.num_hidden_layers           # 32 layers
n_kv_heads = config.num_key_value_heads       # 8 KV heads (GQA)
head_dim = config.hidden_size // config.num_attention_heads  # 128
bytes_per_value = 2                           # FP16

# The formula: 2 (K + V) x layers x kv_heads x head_dim x bytes
kv_per_token = 2 * n_layers * n_kv_heads * head_dim * bytes_per_value

print(f'Model: Mistral-7B-v0.1')
print(f'Layers: {n_layers}')
print(f'KV heads: {n_kv_heads} (GQA, not full 32)')
print(f'Head dimension: {head_dim}')
print(f'Precision: FP16 ({bytes_per_value} bytes)')
print(f'')
print(f'KV per token = 2 x {n_layers} x {n_kv_heads} x {head_dim} x {bytes_per_value}')
print(f'            = {kv_per_token:,} bytes')
print(f'            = {kv_per_token/1024:.0f} KB')

## 2. How KV Cache Scales

131 KB per token sounds small. Multiply by context length and concurrent users:

In [ ]:
# === PARAMETERS (change these to explore) ===
CONTEXT_LENGTHS = [1024, 2048, 4096, 8192, 16384]
USER_COUNTS = [1, 10, 40, 80, 160]

# Print the scaling table
print(f'KV Cache Memory (GB) for Mistral-7B FP16:')
print(f'')
header = f'{"Context":>8}' + ''.join(f'{n:>8} users' for n in USER_COUNTS)
print(header)
print('-' * len(header))

for ctx in CONTEXT_LENGTHS:
    row = f'{ctx:>8}'
    for n_users in USER_COUNTS:
        kv_gb = kv_per_token * ctx * n_users / 1e9
        row += f'{kv_gb:>10.1f}'
    print(row)

print(f'')
print(f'80 users at 4K context = {kv_per_token * 4096 * 80 / 1e9:.1f} GB of KV cache alone.')
print(f'Model weights are another 14.5 GB on top of this.')

In [ ]:
# Visualize: KV cache growth
fig_kv, ax_kv = plt.subplots(figsize=(9, 5))

plot_colors = ['#dbeafe', '#dcfce7', '#fef3c7', '#ffedd5', '#ffe4e6']
for ui, n_users in enumerate(USER_COUNTS):
    kv_values = [kv_per_token * ctx * n_users / 1e9 for ctx in CONTEXT_LENGTHS]
    ax_kv.plot(CONTEXT_LENGTHS, kv_values, 'o-', color=plot_colors[ui],
              linewidth=2, markersize=6, label=f'{n_users} users',
              markeredgecolor='#000', markeredgewidth=0.5)

# GPU available memory lines
ax_kv.axhline(y=57.5, color='#991b1b', linestyle='--', linewidth=1.5,
              label='A100-80 available (57.5 GB)')
ax_kv.axhline(y=118, color='#64748b', linestyle=':', linewidth=1,
              label='H200-141 available (118 GB)')

ax_kv.set_xlabel('Context Length (tokens)', fontsize=11)
ax_kv.set_ylabel('KV Cache Memory (GB)', fontsize=11)
ax_kv.set_title('KV Cache Scales With Users x Context', fontsize=12, fontweight='bold')
ax_kv.legend(fontsize=9, loc='upper left')
ax_kv.spines['top'].set_visible(False)
ax_kv.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Lines above the dashed red = OOM on A100-80.')

## 3. The Capacity Equation

How many concurrent users fit on a given GPU?

```
Available = GPU_VRAM - model_weights - overhead
Max users = Available / (KV_per_token x context_length)
```

In [ ]:
# === PARAMETERS ===
MODEL_WEIGHTS_GB = 14.5   # Mistral-7B FP16
OVERHEAD_GB = 8           # CUDA + activations + framework
CONTEXT = 4096            # tokens per user

# GPU options
GPUS = [
    {'name': 'A100-40',  'vram': 40,  'bw_tbs': 1.6, 'cost_hr': 2.50},
    {'name': 'A100-80',  'vram': 80,  'bw_tbs': 2.0, 'cost_hr': 3.50},
    {'name': 'H100-80',  'vram': 80,  'bw_tbs': 3.4, 'cost_hr': 4.00},
    {'name': 'H200-141', 'vram': 141, 'bw_tbs': 4.8, 'cost_hr': 5.50},
]

# Calculate
kv_per_user_gb = kv_per_token * CONTEXT / 1e9

print(f'Model: Mistral-7B FP16 ({MODEL_WEIGHTS_GB} GB)')
print(f'Context: {CONTEXT} tokens per user')
print(f'KV per user: {kv_per_user_gb*1000:.0f} MB')
print(f'')
print(f'{"GPU":>10} {"VRAM":>6} {"Available":>10} {"Max Users":>10} {"$/hr":>6} {"$/M tok":>8}')
print('-' * 55)

gpu_results = []
for gpu in GPUS:
    available = gpu['vram'] - MODEL_WEIGHTS_GB - OVERHEAD_GB
    max_users = int(available / kv_per_user_gb)
    # Cost: assume each user gets 50 tok/s throughput share
    throughput = max_users * 50  # rough tok/s
    cost_per_m = (gpu['cost_hr'] / 3600) / throughput * 1e6 if throughput > 0 else float('inf')
    gpu_results.append({**gpu, 'available': available, 'max_users': max_users, 'cost_per_m': cost_per_m})
    print(f'{gpu["name"]:>10} {gpu["vram"]:>5}GB {available:>8.1f}GB {max_users:>10} {gpu["cost_hr"]:>5.2f} {cost_per_m:>7.3f}')

print(f'')
print(f'More VRAM = more users = cheaper per token (amortization).')

In [ ]:
# Visualize: cost per million tokens by GPU
fig_gpu, (ax_users, ax_cost) = plt.subplots(1, 2, figsize=(11, 4))

gpu_names = [g['name'] for g in gpu_results]
gpu_max_users = [g['max_users'] for g in gpu_results]
gpu_costs = [g['cost_per_m'] for g in gpu_results]
bar_colors = ['#ffe4e6', '#fef3c7', '#dcfce7', '#dbeafe']

# Max users chart
ax_users.bar(gpu_names, gpu_max_users, color=bar_colors, edgecolor='#000', linewidth=1.2)
ax_users.set_ylabel('Max Concurrent Users')
ax_users.set_title(f'Max Users at {CONTEXT} Context', fontweight='bold')
for ci, cv in enumerate(gpu_max_users):
    ax_users.text(ci, cv + max(gpu_max_users)*0.02, str(cv), ha='center', fontsize=10)
ax_users.spines['top'].set_visible(False)
ax_users.spines['right'].set_visible(False)

# Cost chart
ax_cost.bar(gpu_names, gpu_costs, color=bar_colors, edgecolor='#000', linewidth=1.2)
ax_cost.set_ylabel('$/Million Tokens')
ax_cost.set_title('Cost per Million Tokens (lower = better)', fontweight='bold')
for ci2, cv2 in enumerate(gpu_costs):
    ax_cost.text(ci2, cv2 + max(gpu_costs)*0.02, f'${cv2:.3f}', ha='center', fontsize=10)
ax_cost.spines['top'].set_visible(False)
ax_cost.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'H200 wins despite higher hourly rate: more VRAM = more users = better amortization.')

## Summary

| What | Number | Impact |
|------|--------|--------|
| KV per token | 131 KB | Fixed by model architecture |
| 80 users at 4K | 42 GB | Fills most of A100-80 |
| 80 users at 16K | 168 GB | Exceeds any single GPU |
| Best $/M tokens | H200 | More VRAM = better amortization |

**Key insight:** Weights are fixed. KV cache is the variable cost that determines
how many users you can serve and how much it costs per token.

**To serve more users:** reduce KV cache (quantize, evict, compress) or get more VRAM.
That's what the rest of this workshop teaches.